# 47 creators: followers, engagement and sponsor income

**Spearman's Rank Correlation Matrix** · a worked example from the StatNex teaching guide

**The story.** A marketing agency is choosing social-media creators for a campaign. It collected follower counts, posting frequency, likes, engagement rate and sponsorship income for 47 creators, whose audiences range from 4,100 to over 2 million followers.

**The question.** Do creators with more followers also get more engagement and more sponsorship money? And does posting more often help?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/spearman_correlation.csv` (47 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Creator,Followers,Posts per month,Avg likes per post,Engagement rate (%),Sponsorship income ($/month)
1.0,AvaBakes,32300.0,14.0,2242.0,6.94,200.0
2.0,BenBuilds,66600.0,18.0,4504.0,6.76,530.0
3.0,ChloeStyles,8100.0,30.0,143.0,1.77,30.0
4.0,DevPlays,1378400.0,13.0,34890.0,2.53,6900.0
5.0,EllaRuns,67900.0,21.0,3555.0,5.24,410.0
6.0,FinnFixes,42600.0,21.0,2817.0,6.61,250.0
7.0,GraceGrows,40700.0,3.0,272.0,0.67,140.0
8.0,HugoHikes,32800.0,4.0,439.0,1.34,220.0
9.0,IvyInks,2088300.0,19.0,58651.0,2.81,8790.0
10.0,JackJams,55500.0,2.0,354.0,0.64,390.0
11.0,KaiCodes,335500.0,18.0,35890.0,10.7,800.0
12.0,LunaLifts,74600.0,18.0,4460.0,5.98,360.0
13.0,MiaMakes,4700.0,26.0,232.0,4.95,10.0
14.0,NoahNotes,499700.0,21.0,14793.0,2.96,3060.0
15.0,OliveOutfits,7300.0,21.0,449.0,6.14,20.0
16.0,PaulPaints,2099600.0,27.0,14324.0,0.68,12260.0
17.0,QuinnQuizzes,31400.0,30.0,471.0,1.5,90.0
18.0,RubyReads,498300.0,16.0,27655.0,5.55,1060.0
19.0,SamSaves,461600.0,19.0,29556.0,6.4,2140.0
20.0,TaraTravels,195600.0,14.0,6874.0,3.51,770.0
21.0,UmaUnboxes,31400.0,26.0,1027.0,3.27,110.0
22.0,VicVlogs,4100.0,30.0,144.0,3.52,40.0
23.0,WillWrites,9500.0,18.0,1119.0,11.78,40.0
24.0,XenaExplores,253000.0,20.0,8938.0,3.53,710.0
25.0,YaraYoga,284900.0,11.0,8699.0,3.05,770.0
26.0,ZoeZen,21000.0,14.0,1405.0,6.69,70.0
27.0,AmirArt,771000.0,24.0,18667.0,2.42,1850.0
28.0,BellaBrews,225900.0,23.0,15633.0,6.92,820.0
29.0,CaseyCooks,10000.0,20.0,962.0,9.62,50.0
30.0,DinaDances,126700.0,35.0,547.0,0.43,440.0
31.0,EliEats,121900.0,19.0,6628.0,5.44,260.0
32.0,FayeFits,19700.0,28.0,662.0,3.36,80.0
33.0,GusGames,659800.0,7.0,4624.0,0.7,1490.0
34.0,HanaHome,38800.0,11.0,1051.0,2.71,110.0
35.0,IanInvests,1005800.0,6.0,9503.0,0.94,5470.0
36.0,JunaJewels,129900.0,20.0,7978.0,6.14,820.0
37.0,KofiKicks,2053300.0,16.0,41407.0,2.02,5580.0
38.0,LeoLaughs,1365200.0,23.0,33915.0,2.48,8970.0
39.0,MayaMoney,7800.0,16.0,1634.0,20.95,20.0
40.0,NinaNails,24300.0,18.0,1715.0,7.06,60.0
41.0,OscarOutdoors,630800.0,12.0,19453.0,3.08,2960.0
42.0,PiaPets,46800.0,18.0,1399.0,2.99,180.0
43.0,RoryRides,38900.0,15.0,1924.0,4.95,310.0
44.0,SaraSkincare,141700.0,23.0,7916.0,5.59,580.0
45.0,TomTech,1144200.0,18.0,29611.0,2.59,8940.0
46.0,VeraVintage,482400.0,25.0,7547.0,1.56,1830.0
47.0,WesWorkouts,285100.0,24.0,8912.0,3.13,1110.0

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 160)

## The data

In [ ]:
# One column per variable, one row per case. A case with a blank or non-numeric
# value in any of the variables is dropped from all of them.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Followers", "Posts per month", "Avg likes per post", "Engagement rate (%)", "Sponsorship income ($/month)"]
numeric = data[columns].apply(pd.to_numeric, errors="coerce").dropna()

## Spearman's rank correlation

In [ ]:
# ρ is Pearson's r computed on the ranks (tied values share their average rank). It
# asks whether larger values of one variable go with larger (or smaller) values of
# the other, whether or not the link is a straight line, and it is unchanged by any
# transformation that keeps the order, such as taking logarithms. An extreme value
# counts only as the top rank, so it cannot dominate ρ as it can Pearson's r. Its
# P value (for "ρ = 0") is SciPy's: t = ρ √((n - 2) / (1 - ρ²)) on n - 2 df,
# two-sided, an approximation that is good from about 10 cases.
rho = pd.DataFrame(np.eye(len(columns)), index=columns, columns=columns)
p_values = pd.DataFrame(np.nan, index=columns, columns=columns)
for a, b in combinations(columns, 2):
    result = stats.spearmanr(numeric[a], numeric[b])
    rho.loc[a, b] = rho.loc[b, a] = result.statistic
    p_values.loc[a, b] = p_values.loc[b, a] = result.pvalue
sizes = pd.DataFrame(len(numeric), index=columns, columns=columns)
print(rho)

## Test of the rank correlation coefficients

In [ ]:
# The app prints the P value above the diagonal and * (P < 0.05) / ** (P < 0.01)
# below it. The tests assume independent cases. A matrix of k variables runs
# k(k - 1)/2 tests, and about 1 in 20 of them shows P < 0.05 by chance alone, so a
# lone starred pair is weak evidence; read ρ for the strength.
print()
print(p_values)
print(f"\nn = {len(numeric)}")

## What the result says

Spearman's rho is a rank correlation. Its sign shows whether one measure keeps rising (plus) or keeps falling (minus) as the other rises. Followers and sponsorship income have rho = 0.98: bigger creators earn more, almost without exception.

Followers and engagement rate have rho = -0.40 with P = 0.005. The P value, the chance of a link this strong if there were none, is small. So a smaller share of a big creator's followers engages with each post.

Posts per month shows no rank correlation with anything: every rho is between -0.15 and 0.02, and the smallest P value is 0.326. That does not prove posting has no effect. Rho only sees links that keep rising or keep falling.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.